# Finance AI Agent using OpenAI Agents SDK + Ollama

## Objective

In this notebook, we will build a **Finance AI Agent** that can answer stock market
questions using:

- **OpenAI Agents SDK** – gives our AI a way to call Python functions ("tools")
- **Ollama** – runs a large language model (LLM) locally on our own machine
- **Yahoo Finance (`yfinance`)** – provides live stock market data for free
- **Gradio** – gives our agent a simple web page (UI) to chat with

## What the Agent Can Do

By the end of this notebook, our agent will be able to:

1. Look up the **current price** of a stock
2. Show the latest **analyst recommendations** for a stock
3. Read the latest **news** about a stock and summarize its **sentiment**
   (positive / negative / neutral)
4. Do all of this through a simple **chat-style web page**, running entirely on
   our local computer (no data sent to the cloud)

## How the Notebook is Organized

We will build this step by step:

1. Setup — configure the connection to our local Ollama model
2. Import the libraries we need
3. Turn off cloud-based tracing/logging (not needed for local use)
4. Build our first "tool" — get the current stock price
5. Build our second "tool" — get analyst recommendations
6. Explore the news data returned by Yahoo Finance
7. Build our third "tool" — get and summarize stock news
8. Create two AI agents that use these tools
9. Write helper functions to run both agents together
10. Build and launch a simple chat web page using Gradio

---

## Step 1: Connect to our Local Ollama Model

The OpenAI Agents SDK normally talks to OpenAI's cloud servers. Since we want to
run everything **locally and for free** using **Ollama**, we need to tell the SDK
two things:

- **Where to send requests** → our local Ollama server (`http://localhost:11434/v1`)
- **What API key to use** → Ollama doesn't need a real key, so we just use the
  placeholder text `"ollama"`

> **Note:** Make sure Ollama is installed and running on your computer before
> running this notebook. You can check by opening `http://localhost:11434` in
> your browser.

In [1]:
# We use the "os" library to set environment variables.
# Environment variables are a way to store settings that our program will read.
import os

# Tell the OpenAI Agents SDK to send requests to our local Ollama server
# instead of the real OpenAI cloud servers.
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"

# Ollama does not check for a real API key, so we just use a placeholder value.
os.environ["OPENAI_API_KEY"] = "ollama"


## Step 2: Import the Libraries We Need

Here we import all the tools (Python libraries) required for this project:

- `yfinance` → to download real stock market data from Yahoo Finance
- `agents` (OpenAI Agents SDK) → to create AI agents and give them tools
- `asyncio` → to run multiple agents at the same time (in parallel)
- `gradio` → to build a simple web page for chatting with our agents

In [2]:
# Library to fetch stock market data from Yahoo Finance
import yfinance as yf

# Core building blocks from the OpenAI Agents SDK
from agents import (
    Agent,              # Used to create an AI agent
    Runner,             # Used to run an agent and get its response
    function_tool,      # Decorator that turns a normal Python function into a tool
    set_tracing_disabled # Used to turn off cloud tracing/logging
)

# Library that lets us run multiple tasks (like our two agents) at the same time
import asyncio

# Library to quickly build a simple web-based chat interface
import gradio as gr


D:\Tech Project\Finace AI Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Disable Cloud Tracing

The OpenAI Agents SDK can send "traces" (logs of what the agent is doing) to
OpenAI's cloud dashboard. This is useful when using the real OpenAI API, but
since we are running everything **locally with Ollama**, this cloud tracing
is unnecessary. We disable it below.

In [3]:
# Turn off tracing so the SDK does not try to send logs to OpenAI's cloud
set_tracing_disabled(True)


## Step 4: Tool 1 — Get Current Stock Price

Our AI agent cannot fetch live stock prices on its own — LLMs only know text
patterns, not real-time data. So we give it a **tool**: a normal Python
function that the agent can call whenever it needs a stock price.

**Input:** A ticker symbol (for example: `TSLA`, `AAPL`, `MSFT`)

**Output:** A text message containing the latest stock price

The `@function_tool` decorator (from the Agents SDK) is what makes this
regular Python function usable by our AI agent.

In [4]:
# ============================================================
# Tool: Get Current Stock Price
# ============================================================

@function_tool
def get_stock_price(ticker: str) -> str:
    """
    Fetch the latest closing stock price for the given ticker symbol.
    """

    # Create a Yahoo Finance object for the given ticker (e.g. "TSLA")
    stock = yf.Ticker(ticker)

    # Download the last 1 day of price history for this stock
    history = stock.history(period="1d")

    # Get the closing price from the very last row of data
    latest_close = history["Close"].iloc[-1]

    # Return a simple, readable text message with the price
    return f"The current price of {ticker} is ${latest_close:.2f}"


## Step 5: Tool 2 — Analyst Recommendations

This tool lets our agent fetch **analyst ratings** (Buy / Hold / Sell etc.)
for a stock from Yahoo Finance.

- If recommendations are available, we return the **10 most recent** ones.
- If no recommendations exist for that ticker, we return a friendly message
  instead of an error.

In [5]:
# ============================================================
# Tool: Get Analyst Recommendations
# ============================================================

@function_tool
def get_analyst_recommendations(ticker: str) -> str:
    """
    Retrieve analyst recommendations for the specified stock ticker.
    """

    # Create a Yahoo Finance object for the given ticker
    stock = yf.Ticker(ticker)

    # Download the analyst recommendation data (as a table/DataFrame)
    recommendations = stock.recommendations

    # If there is no data available, tell the user instead of returning an error
    if recommendations is None or recommendations.empty:
        return f"No analyst recommendations found for {ticker}"

    # Keep only the 10 most recent recommendations
    latest_recommendations = recommendations.tail(10)

    # Convert the table into a nicely formatted markdown table (text)
    return latest_recommendations.to_markdown()


## Step 6: Explore the News Data (Optional)

Before writing our news tool, it helps to look at the **raw data** that
`yfinance` gives us for stock news. This way we know exactly which fields
(like `title`, `summary`, `publisher`) we need to pull out.

This cell is just for exploration — it prints one sample news item for
Tesla (`TSLA`) so we can see its structure.

In [6]:
# Look at the raw structure of one news item for TSLA.
# This helps us understand what fields are available before building our tool.
stock = yf.Ticker("TSLA")
print(stock.news[:1])


[{'id': '7b99391e-9228-43c1-b448-6e9d8d43b527', 'content': {'id': '7b99391e-9228-43c1-b448-6e9d8d43b527', 'contentType': 'VIDEO', 'title': 'YieldMax launches a new SpaceX options ETF: What investors should know', 'description': '<p>SpaceX (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/SPCX/" data-i13n="slk:SPCX;cpos:1;pos:1">SPCX</a>) dropped below its IPO price on Wednesday, sinking below $140 per share.</p>\n<p>YieldMax ETFs chief strategist Mike Khouw comes on Market Domination to discuss his firm\'s new ETF product with options-based strategies on SpaceX: the YieldMax SPCX Option Income Strategy ETF (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/YSPC/" data-i13n="cpos:2;pos:1">YSPC</a>).</p>', 'summary': "SpaceX (SPCX) dropped below its IPO price on Wednesday, sinking below $140 per share. YieldMax ETFs chief strategist Mike Khouw comes on Market Domination to discuss his firm's new ETF product with options-based strateg

## Step 7: Tool 3 — Get Stock News

Using what we learned from exploring the data above, we can now build a tool
that fetches the **latest news headlines and summaries** for a given stock.

This tool will later be used by our **News Sentiment Agent** to judge whether
the news about a stock is positive, negative, or mixed.

In [7]:
# ============================================================
# Tool: Get Stock News
# ============================================================

@function_tool
def get_stock_news(ticker: str) -> str:
    """
    Fetch latest news headlines and summaries for the given stock ticker.
    """

    # Create a Yahoo Finance object and fetch its news list
    stock = yf.Ticker(ticker)
    news_items = stock.news

    # If there is no news at all, return a friendly message
    if not news_items:
        return f"No recent news found for {ticker}"

    # This list will hold the formatted text for each news article
    articles = []

    # Only look at the first 10 news items
    for item in news_items[:10]:
        # Each news item stores its details inside a "content" dictionary
        content = item.get("content", {})

        # Pull out the fields we care about, with safe defaults if missing
        title = content.get("title", "No title")
        summary = content.get("summary", "")
        provider = content.get("provider", {})
        publisher = provider.get("displayName", "Unknown source")

        # Skip any article that has no real title (likely bad/incomplete data)
        if title == "No title":
            continue

        # Format the article as "- [Publisher] Title \n  Summary"
        articles.append(f"- [{publisher}] {title}\n  {summary}")

    # If nothing useful was collected, say so
    if not articles:
        return f"No recent news found for {ticker}"

    # Join all articles into one big text block, separated by new lines
    return "\n".join(articles)


## Step 8: Create the AI Agents

Now that our tools are ready, we create two separate AI agents:

1. **Finance Agent** — answers questions about stock **price** and
   **analyst recommendations**, and presents data in tables when useful.
2. **News Sentiment Agent** — always looks up the latest **news** for a
   stock and summarizes whether the overall sentiment is Positive,
   Negative, or Mixed/Neutral.

Both agents run on the same local model (`gemma4:31b-cloud` served through
Ollama), but each agent is given a different set of **tools** and different
**instructions**, so they behave differently.

In [8]:
# ============================================================
# Create the Finance AI Agents
# ============================================================

# This agent focuses on stock price and analyst recommendations
finance_agent = Agent(
    name="Finance Agent",
    instructions=(
        "You are a helpful finance assistant. "
        "Whenever appropriate, present structured information using tables."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_price, get_analyst_recommendations],
)

# This agent focuses on reading news and judging overall sentiment
news_sentiment_agent = Agent(
    name="News Sentiment Agent",
    instructions=(
        "You ALWAYS analyze stock news sentiment immediately - never ask for "
        "permission or confirmation. "
        "When given any query mentioning a company or ticker, immediately call "
        "get_stock_news for that ticker. "
        "Do not ask the user if they want news analysis - just do it. "
        "Steps: "
        "1. Call get_stock_news with the ticker extracted from the query. "
        "2. Judge overall tone: Positive, Negative, or Mixed/Neutral. "
        "3. Give a short summary (3-5 bullet points) in simple, plain English. "
        "4. End with one line: 'Overall Sentiment: <Positive/Negative/Mixed>'. "
        "Never respond with a question. Always respond with the analysis."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_news],
)


## Step 9: Run Both Agents Together

We want the **Finance Agent** and the **News Sentiment Agent** to answer the
same question at the same time, instead of waiting for one to finish before
starting the other. We do this using `asyncio`, which lets Python run
multiple tasks **in parallel**.

We write two small helper functions:

1. `run_both_agents` — an **async** function that starts both agents at once
   and waits for both to finish.
2. `chat_with_agents` — a normal (non-async) function that Gradio can call.
   It simply runs `run_both_agents` behind the scenes.

In [9]:
# This function runs both agents at the same time and waits for both results
async def run_both_agents(user_input: str):
    # Start the Finance Agent (this does not block/wait yet)
    price_task = Runner.run(finance_agent, user_input)

    # Start the News Sentiment Agent (this also does not block/wait yet)
    sentiment_task = Runner.run(news_sentiment_agent, user_input)

    # Now wait for both agents to finish, running them in parallel
    price_result, sentiment_result = await asyncio.gather(price_task, sentiment_task)

    # Return the final text answer from each agent
    return price_result.final_output, sentiment_result.final_output


In [10]:
# Gradio does not work directly with "async" functions, so we create a
# normal function that runs our async function for us using asyncio.run().
def chat_with_agents(user_input: str):
    price_output, sentiment_output = asyncio.run(run_both_agents(user_input))
    return price_output, sentiment_output


## Step 10: Build and Launch the Web Interface

Finally, we use **Gradio** to create a simple chat-style web page where a
user can type a question (for example: *"What is the current price of
TSLA?"*) and see two separate answers side by side:

1. **Stock Price / Recommendations** — from the Finance Agent
2. **News Sentiment** — from the News Sentiment Agent

Running the cell below will start a local web server and open the app in
your browser.

In [11]:
# Build the web interface using Gradio
demo = gr.Interface(
    fn=chat_with_agents,  # the function to call when the user submits a question
    inputs=gr.Textbox(
        label="Ask about a stock",
        placeholder="e.g. What is the current price of TSLA?"
    ),
    outputs=[
        gr.Textbox(label="Stock Price / Recommendations", lines=10),
        gr.Textbox(label="News Sentiment", lines=10),
    ],
    title="Finance AI Agent",
    description="Get stock price info and public sentiment side by side.",
)

# Launch the web app (this will open a local URL you can visit in your browser)
demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
